<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Quantum_Toroids_and_Spinors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Spinor Double-Valued Return Visualization

## Overview
This notebook contains the implementation of a high-fidelity mathematical animation using the Manim library. The goal is to visualize the topological and algebraic properties of spinors, specifically demonstrating why a 360-degree rotation results in a sign flip and why a 720-degree rotation is required to return a system to its original state.

## Technical Background

### The Spinor Concept
In physics and mathematics, spinors are elements of a complex vector space that can be associated with Euclidean space. Unlike vectors, which return to their original state after a 360-degree (2π) rotation, spinors exhibit a property known as 'double-valuedness'.

### Geometric Interpretation
When an object is rotated through 360 degrees, its orientation in space is identical to its starting position. However, the path taken through the rotation group (SO(3)) is not contractible to a point. The universal cover of the rotation group is SU(2), which is topologically a 3-sphere. A 360-degree rotation corresponds to a path from the identity to the negative identity in SU(2). Only a 720-degree (4π) rotation corresponds to a closed loop in SU(2) that can be continuously shrunk to a point.

### Toroidal Mapping
This animation uses a torus to map the phase space. By traversing the major and minor radii of the torus, we can represent the geometric rotation alongside the internal phase evolution (the spinor state). The visualization highlights:
1. The initial state (psi).
2. The state after 360 degrees (negative psi), where the position on the torus returns to the same longitude but on the opposite side of the surface.
3. The state after 720 degrees (positive psi), where both the geometric position and the internal phase are restored.

## Implementation Details
- Library: Manim (Community Edition)
- Resolution: 720x1280 (Vertical/Shorts format)
- Key Techniques: Custom parametric surfaces, ValueTracker-based state synchronization, and wobbled Bezier paths for a hand-drawn aesthetic.

In [3]:
!apt-get update -qq

# 1. Create a local fonts directory
!mkdir -p ~/.fonts

# 2. Download a guaranteed irregular, handwritten font (Caveat)
!wget -qO ~/.fonts/Caveat-Regular.ttf https://github.com/googlefonts/caveat/raw/main/fonts/ttf/Caveat-Regular.ttf
!wget -qO ~/.fonts/Caveat-Bold.ttf https://github.com/googlefonts/caveat/raw/main/fonts/ttf/Caveat-Bold.ttf

# 3. Refresh the Linux system font cache so Manim/Pango can see it immediately
!fc-cache -fv

!apt-get install -y -qq libcairo2-dev libpango1.0-dev ffmpeg dvisvgm texlive-latex-extra texlive-fonts-extra
!pip install -q manim


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 2 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no such directory
/root/.fonts: caching, new cache contents: 2 fonts, 0 dirs
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/humor-sans: skipping, looped directory detected
/usr/share/fonts/truetype/liberation: skipping, looped directory detected
/var/cache/fontconfig: cleaning cache directory
/root/.cache/fontconfig: not cleaning non-existent cache directory
/

In [1]:
import manim
from manim.utils.ipython_magic import ManimMagic

try:
    # Manually register the %%manim magic command into the IPython shell
    get_ipython().register_magics(ManimMagic)
    print(f"Manim {manim.__version__} loaded and magic commands registered successfully.")
except Exception as e:
    print(f"Error loading Manim magic: {e}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Manim 0.20.1 loaded and magic commands registered successfully.


In [5]:
%%manim -v WARNING -r 720,1280 --fps 24 SpinorDoubleReturnScene



"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Double-Valued Return
Repo: github.com/zombimann/Mathematical-video-animations-and-visualization
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
from manim import *

config.pixel_width = 720
config.pixel_height = 1280
config.frame_width = 9
config.frame_height = 16
config.frame_rate = 24
config.background_color = "#100D0A"


@dataclass(frozen=True)
class VideoDesign:
    width: float = 9
    height: float = 16
    fps: int = 24
    major_radius: float = 2.00
    minor_radius: float = 0.62
    intro_seconds: float = 2.2
    traversal_seconds: float = 11.8
    pause_seconds: float = 2.1
    reveal_seconds: float = 2.2
    closing_seconds: float = 1.55


DESIGN = VideoDesign()
OUTPUT_FILE = Path("spinor_double_return_shorts.mp4")
MANIM_OUTPUT = Path(
    "media/videos/spinor_double_return/1280p24/SpinorDoubleReturnScene.mp4"
)


PALETTE = dict(
    night="#100D0A",
    paper="#D8C08F",
    paper_dark="#6F5638",
    ink="#F7EEDC",
    ink_soft="#E0CFB0",
    panel="#22180F",
    panel_stroke="#DECBA6",
    gold="#F5C96F",
    cyan="#8ED8EA",
    blue="#426F86",
    red="#D96B5F",
)

# Using 'Lobster Two' from the provided available system fonts list
HAND_FONT = "Lobster Two"
WATERMARK = "© Mugambi Ndwiga / @craftsandengineering"


def script_note() -> str:
    return (
        "Story plan: start on the top of a torus; after one 360-degree sweep the "
        "marker reaches the same longitude on the underside and the spinor is -psi; "
        "after the second sweep it returns to the top and the state is restored."
    )


def hand_text(message: str, size: int = 26, color: str = PALETTE["ink"]) -> Text:
    # Removed disable_ligatures=True to fix the trailing text truncation bug
    return Text(message, font=HAND_FONT, font_size=size, color=color)


def apply_wobble(mobject: Mobject, scale: float = 0.003, seed: int = 42) -> Mobject:
    """Perturbs VMobject Bezier points slightly to create an irregular hand-drawn look."""
    rng = np.random.default_rng(seed)
    for submob in mobject.get_family():
        if isinstance(submob, VMobject) and not isinstance(submob, (Text, Paragraph, MarkupText)):
            if len(submob.points) > 0:
                points = submob.points
                noise = rng.uniform(-scale, scale, size=(len(points), 3))
                noise[:, 2] = 0.0  # Keep it constrained to 2D
                submob.points = points + noise
    return mobject


def soft_panel(width: float, height: float, center: np.ndarray, opacity: float = 0.30) -> RoundedRectangle:
    panel = RoundedRectangle(
        corner_radius=0.10,
        width=width,
        height=height,
        fill_color=PALETTE["panel"],
        fill_opacity=opacity,
        stroke_color=PALETTE["panel_stroke"],
        stroke_opacity=0.58,
        stroke_width=1.4,
    ).move_to(center)
    return apply_wobble(panel, scale=0.035)


def fade_peak(value: float, center: float, half_width: float) -> float:
    return float(np.clip(1.0 - abs(value - center) / half_width, 0.0, 1.0))


def rotation_about(angle: float, axis: np.ndarray) -> np.ndarray:
    return rotation_matrix(angle, axis)


class SpinorDoubleReturnScene(Scene):
    """A 9:16 Manim short explaining why spinor states restore after 720 degrees."""

    def construct(self) -> None:
        self.camera.background_color = PALETTE["night"]
        self._build_manuscript_backdrop()
        self._build_fixed_layout()
        self._build_animation()

    def _build_manuscript_backdrop(self) -> None:
        rng = np.random.default_rng(12)
        paper_wash = Rectangle(
            width=DESIGN.width,
            height=DESIGN.height,
            fill_color=PALETTE["paper"],
            fill_opacity=0.075,
            stroke_width=0,
        )
        vignette = Rectangle(
            width=DESIGN.width,
            height=DESIGN.height,
            fill_color="#070504",
            fill_opacity=0.20,
            stroke_width=0,
        )
        ruled_lines = VGroup(
            *[
                Line(
                    [-4.2, y, 0],
                    [4.2, y, 0],
                    color=PALETTE["blue"],
                    stroke_width=0.7,
                    stroke_opacity=0.12,
                )
                for y in np.linspace(-6.4, 6.2, 14)
            ]
        )
        apply_wobble(ruled_lines, scale=0.03)

        blotches = VGroup()
        for _ in range(30):
            blotch = Ellipse(
                width=rng.uniform(0.28, 1.65),
                height=rng.uniform(0.15, 0.95),
                fill_color=PALETTE["paper_dark"],
                fill_opacity=rng.uniform(0.025, 0.075),
                stroke_width=0,
            )
            blotch.move_to([rng.uniform(-4.2, 4.2), rng.uniform(-7.5, 7.5), 0])
            blotch.rotate(rng.uniform(-PI, PI))
            blotches.add(blotch)

        self.add(paper_wash, ruled_lines, blotches, vignette)

    def _build_fixed_layout(self) -> None:
        self.title_band = soft_panel(7.95, 0.72, np.array([0, 7.20, 0]), 0.36)
        self.torus_panel = soft_panel(8.38, 5.54, np.array([0, 2.52, 0]), 0.27)
        self.spinor_panel = soft_panel(8.38, 5.95, np.array([0, -3.35, 0]), 0.27)
        self.footer_band = soft_panel(7.95, 0.58, np.array([0, -7.25, 0]), 0.34)
        self.add(
            self.title_band,
            self.torus_panel,
            self.spinor_panel,
            self.footer_band,
        )

        self.title_text = hand_text("One loop is not enough", 29).move_to([0, 7.20, 0])
        torus_caption = hand_text("Toroidal traversal", 22).move_to([0, 4.92, 0])
        spinor_caption = hand_text("Spinor phase response", 22).move_to([0, -0.74, 0])
        footer = hand_text("State:  ψ(θ) = exp(iθ/2)  ·  θ = geometric rotation", 17)
        footer.move_to([0, -7.25, 0])
        watermark = hand_text(WATERMARK, 14).set_opacity(0.60)
        watermark.to_corner(DR, buff=0.18).shift(UP * 0.05)
        self.add(
            self.title_text,
            torus_caption,
            spinor_caption,
            footer,
            watermark,
        )

    def _build_animation(self) -> None:
        journey = ValueTracker(0.0)
        torus_center = np.array([0.0, 2.38, 0.0])
        dial_center = np.array([0.0, -3.34, 0.0])
        torus_orientation = rotation_about(58 * DEGREES, RIGHT) @ rotation_about(-32 * DEGREES, OUT)

        def torus_local(u: float, v: float) -> np.ndarray:
            return np.array(
                [
                    (DESIGN.major_radius + DESIGN.minor_radius * np.cos(v)) * np.cos(u),
                    (DESIGN.major_radius + DESIGN.minor_radius * np.cos(v)) * np.sin(u),
                    DESIGN.minor_radius * np.sin(v),
                ]
            )

        def route_u(progress: float) -> float:
            return TAU * progress

        def route_v(progress: float) -> float:
            return PI / 2 + PI * progress

        def route_point(progress: float) -> np.ndarray:
            return world_point(route_u(progress), route_v(progress))

        def world_point(u: float, v: float) -> np.ndarray:
            point = torus_orientation @ torus_local(u, v)
            point[1] *= 0.96
            point[2] *= 0.34
            return point + torus_center

        def world_vector(vec: np.ndarray) -> np.ndarray:
            projected = torus_orientation @ vec
            projected[1] *= 0.96
            projected[2] *= 0.34
            return projected

        def frame_vectors(u: float, spinor_phase: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
            radial = np.array([np.cos(u), np.sin(u), 0.0])
            tangent = np.array([-np.sin(u), np.cos(u), 0.0])
            vertical = np.array([0.0, 0.0, 1.0])
            normal = np.cos(spinor_phase) * radial + np.sin(spinor_phase) * vertical
            binormal = np.cross(tangent, normal)
            binormal = binormal / np.linalg.norm(binormal)
            return tangent, normal, binormal

        torus = self._projected_torus(world_point, torus_center)

        route_shadow = ParametricFunction(
            lambda t: route_point(t),
            t_range=[0, 2],
            color=PALETTE["gold"],
            stroke_width=14,
            stroke_opacity=0.08,
        )
        first_lap_route = ParametricFunction(
            lambda t: route_point(t),
            t_range=[0, 1],
            color=PALETTE["gold"],
            stroke_width=6.4,
            stroke_opacity=0.52,
        )
        second_lap_route = ParametricFunction(
            lambda t: route_point(t),
            t_range=[1, 2],
            color=PALETTE["cyan"],
            stroke_width=5.8,
            stroke_opacity=0.42,
        )

        apply_wobble(route_shadow, scale=0.02)
        apply_wobble(first_lap_route, scale=0.02)
        apply_wobble(second_lap_route, scale=0.02)

        marker = always_redraw(
            lambda: Dot(
                route_point(journey.get_value()),
                radius=0.086 + 0.022 * (1 + np.sin(route_v(journey.get_value()))) / 2,
                color=PALETTE["gold"],
                fill_opacity=0.78 + 0.20 * (1 + np.sin(route_v(journey.get_value()))) / 2,
            )
        )
        marker_halo = always_redraw(
            lambda: Dot(
                route_point(journey.get_value()),
                radius=0.190,
                color=PALETTE["gold"],
                fill_opacity=0.09 + 0.08 * (1 + np.sin(route_v(journey.get_value()))) / 2,
                stroke_width=0,
            )
        )

        def local_frame() -> VGroup:
            progress = journey.get_value()
            u = route_u(progress)
            tangent, normal, binormal = frame_vectors(u, PI * progress)
            origin = route_point(progress)
            frame = VGroup(
                Line(origin, origin + world_vector(tangent * 0.54), color=PALETTE["gold"], stroke_width=4.4),
                Line(origin, origin + world_vector(normal * 0.50), color=PALETTE["ink"], stroke_width=3.9),
                Line(origin, origin + world_vector(binormal * 0.50), color=PALETTE["cyan"], stroke_width=3.9),
            )
            return apply_wobble(frame, scale=0.015)

        frame_axes = always_redraw(local_frame)
        start_point = route_point(0)
        underside_point = route_point(1)
        depth_line = DashedLine(
            start_point,
            underside_point,
            dash_length=0.08,
            color=PALETTE["ink_soft"],
            stroke_width=2.0,
            stroke_opacity=0.55,
        )
        apply_wobble(depth_line, scale=0.02)
        depth_line.set_opacity(0)
        depth_line.add_updater(lambda mob: mob.set_opacity(fade_peak(journey.get_value(), 1.0, 0.24)))

        start_label = hand_text("top start", 17, PALETTE["ink_soft"]).move_to(start_point + UP * 0.46 + LEFT * 0.18)
        underside_label = hand_text("same longitude, underside", 17, PALETTE["red"]).move_to(
            underside_point + DOWN * 0.44
        )
        underside_label.set_opacity(0)
        underside_label.add_updater(lambda mob: mob.set_opacity(fade_peak(journey.get_value(), 1.0, 0.30)))

        self.add(
            torus,
            route_shadow,
            first_lap_route,
            second_lap_route,
            depth_line,
            start_label,
            underside_label,
            marker_halo,
            marker,
            frame_axes,
        )

        dial = self._spinor_dial(dial_center, journey)
        self.add(dial)

        lap_counter = self._lap_counter("first circuit", 0).move_to([0, 0.25, 0])
        state_badge = self._state_badge("state = ψ", PALETTE["ink_soft"]).move_to([0, -6.05, 0])
        self.add(lap_counter, state_badge)

        self.wait(DESIGN.intro_seconds)
        self.play(journey.animate.set_value(1.0), run_time=DESIGN.traversal_seconds, rate_func=smooth)
        self._retitle("360°: same longitude, opposite side")
        new_lap_counter = self._lap_counter("first circuit complete", 1).move_to(lap_counter.get_center())
        new_state_badge = self._state_badge("state = −ψ", PALETTE["red"]).move_to(state_badge.get_center())
        self.play(
            FadeTransform(lap_counter, new_lap_counter),
            FadeTransform(state_badge, new_state_badge),
            run_time=0.32,
        )
        lap_counter = new_lap_counter
        state_badge = new_state_badge
        self.wait(DESIGN.pause_seconds)
        self.play(journey.animate.set_value(2.0), run_time=DESIGN.traversal_seconds, rate_func=smooth)
        self._retitle("720°: back to the top")
        final_lap_counter = self._lap_counter("two circuits complete", 2).move_to(lap_counter.get_center())
        final_state_badge = self._state_badge("state = +ψ", PALETTE["gold"]).move_to(state_badge.get_center())
        self.play(
            FadeTransform(lap_counter, final_lap_counter),
            FadeTransform(state_badge, final_state_badge),
            run_time=0.32,
        )
        self.wait(DESIGN.reveal_seconds)
        self._closing_card()

    def _projected_torus(self, world_point, center: np.ndarray) -> VGroup:
        body = VGroup()
        tube_wash = Ellipse(
            width=5.82,
            height=3.26,
            fill_color="#24394B",
            fill_opacity=0.18,
            stroke_color=PALETTE["panel_stroke"],
            stroke_opacity=0.16,
            stroke_width=1.1,
        ).move_to(center)
        hole = Ellipse(
            width=2.36,
            height=1.18,
            fill_color=PALETTE["night"],
            fill_opacity=0.62,
            stroke_color=PALETTE["ink_soft"],
            stroke_opacity=0.15,
            stroke_width=1.0,
        ).move_to(center)
        body.add(tube_wash)

        # Made geometry grid lines significantly more visible by increasing width and opacity
        for v in np.linspace(0, TAU, 13, endpoint=False):
            curve = ParametricFunction(
                lambda u, vv=v: world_point(u, vv),
                t_range=[0, TAU],
                color=PALETTE["blue"],
                stroke_width=1.6,
                stroke_opacity=0.55,
            )
            body.add(curve)

        for u in np.linspace(0, TAU, 10, endpoint=False):
            curve = ParametricFunction(
                lambda v, uu=u: world_point(uu, v),
                t_range=[0, TAU],
                color=PALETTE["ink_soft"],
                stroke_width=1.4,
                stroke_opacity=0.45,
            )
            body.add(curve)

        body.add(hole)
        apply_wobble(body, scale=0.02)
        return body

    def _spinor_dial(self, center: np.ndarray, journey: ValueTracker) -> VGroup:
        dial_radius = 1.45

        def arrow() -> Arrow:
            phase = PI * journey.get_value()
            arr = Arrow(
                start=LEFT * 1.02,
                end=RIGHT * 1.02,
                buff=0,
                color=PALETTE["gold"],
                stroke_width=9,
                max_tip_length_to_length_ratio=0.23,
            ).rotate(phase).move_to(center)
            return apply_wobble(arr, scale=0.03)

        def phase_dot() -> Dot:
            phase = PI * journey.get_value()
            return Dot(
                center
                + 1.78
                * np.array(
                    [
                        np.cos(phase),
                        np.sin(phase),
                        0.0,
                    ]
                ),
                radius=0.055,
                color=PALETTE["cyan"],
            )

        def minus_badge() -> VGroup:
            badge = VGroup(
                Circle(
                    radius=0.29,
                    color=PALETTE["red"],
                    stroke_width=2.2,
                    fill_color=PALETTE["red"],
                    fill_opacity=0.14,
                ),
                Line(LEFT * 0.105, RIGHT * 0.105, color=PALETTE["red"], stroke_width=3.0),
            )
            badge.move_to(center + DOWN * 1.72)
            return apply_wobble(badge, scale=0.02)

        def plus_badge() -> VGroup:
            badge = VGroup(
                Circle(
                    radius=0.27,
                    color=PALETTE["gold"],
                    stroke_width=2,
                    fill_color=PALETTE["gold"],
                    fill_opacity=0.10,
                ),
                Line(LEFT * 0.095, RIGHT * 0.095, color=PALETTE["gold"], stroke_width=2.8),
                Line(DOWN * 0.095, UP * 0.095, color=PALETTE["gold"], stroke_width=2.8),
            )
            badge.move_to(center + DOWN * 1.72)
            return apply_wobble(badge, scale=0.02)

        c1 = Circle(radius=1.78, color=PALETTE["blue"], stroke_width=2.2, stroke_opacity=0.42).move_to(center)
        c2 = Circle(radius=dial_radius, color=PALETTE["panel_stroke"], stroke_width=3.0, stroke_opacity=0.70).move_to(center)
        l1 = Line(center + LEFT * 1.20, center + RIGHT * 1.20, color=PALETTE["ink_soft"], stroke_width=1.6, stroke_opacity=0.23)
        l2 = Line(center + UP * 1.20, center + DOWN * 1.20, color=PALETTE["ink_soft"], stroke_width=1.6, stroke_opacity=0.23)

        apply_wobble(c1, scale=0.03)
        apply_wobble(c2, scale=0.03)
        apply_wobble(l1, scale=0.03)
        apply_wobble(l2, scale=0.03)

        dial_group = VGroup(
            c1,
            c2,
            l1,
            l2,
            hand_text("ψ", 30).move_to(center + UP * 2.12),
            always_redraw(arrow),
            always_redraw(phase_dot),
            minus_badge(),
            plus_badge(),
        )
        dial_group[-2].add_updater(lambda mob: mob.set_opacity(fade_peak(journey.get_value(), 1.0, 0.72)))
        dial_group[-1].add_updater(
            lambda mob: mob.set_opacity(
                max(fade_peak(journey.get_value(), 0.0, 0.34), fade_peak(journey.get_value(), 2.0, 0.42))
            )
        )
        return dial_group

    def _lap_counter(self, label_text: str, completed_laps: int) -> VGroup:
        label = hand_text(label_text, 18, PALETTE["ink_soft"])
        dots = VGroup()
        for index in range(2):
            filled = completed_laps > index
            circ = Circle(
                radius=0.105,
                stroke_color=PALETTE["gold"],
                stroke_width=1.5,
                fill_color=PALETTE["gold"],
                fill_opacity=0.72 if filled else 0.08,
            )
            apply_wobble(circ, scale=0.015)
            dots.add(circ)
        dots.arrange(RIGHT, buff=0.15)
        group = VGroup(label, dots).arrange(DOWN, buff=0.12)
        return group

    def _state_badge(self, message: str, color: str) -> VGroup:
        badge = soft_panel(2.15, 0.56, np.array([0, 0, 0]), opacity=0.28)
        label = hand_text(message, 20, color)
        return VGroup(badge, label)

    def _retitle(self, message: str) -> None:
        title_size = 24 if len(message) > 28 else 28
        new_title = hand_text(message, title_size).move_to(self.title_text.get_center())
        self.play(FadeTransform(self.title_text, new_title), run_time=0.42)
        self.title_text = new_title

    def _closing_card(self) -> None:
        background = Rectangle(
            width=DESIGN.width,
            height=DESIGN.height,
            fill_color=PALETTE["night"],
            fill_opacity=1.0,
            stroke_width=0,
        )
        stain = Ellipse(
            width=5.5,
            height=2.25,
            fill_color=PALETTE["paper_dark"],
            fill_opacity=0.11,
            stroke_width=0,
        ).rotate(-14 * DEGREES)
        title = hand_text("Double-valued return", 37).move_to([0, 1.85, 0])
        subtitle = hand_text("One loop flips the sign; two restore it.", 23).move_to([0, 0.96, 0])
        made_by = hand_text("Made by Mugambi Ndwiga", 25).move_to([0, -0.46, 0])
        handle = hand_text("@craftsandengineering", 22, PALETTE["ink_soft"]).move_to([0, -1.08, 0])
        mark = hand_text(WATERMARK, 14).set_opacity(0.60).to_corner(DR, buff=0.18).shift(UP * 0.05)
        self.add(background, stain, title, subtitle, made_by, handle, mark)
        self.wait(DESIGN.closing_seconds)


Manim Community v0.20.1